In [ ]:
from pathlib import Path
import runpy

BOOTSTRAP_CANDIDATES = (
    "notebooks/_bootstrap.py",
    "abstractgraph/notebooks/_bootstrap.py",
    "abstractgraph-ml/notebooks/_bootstrap.py",
    "abstractgraph-generative/notebooks/_bootstrap.py",
    "abstractgraph-graphicalizer/notebooks/_bootstrap.py",
)

_bootstrap_path = next(
    (
        candidate / relative
        for candidate in (Path.cwd(), *Path.cwd().parents)
        for relative in BOOTSTRAP_CANDIDATES
        if (candidate / relative).exists()
    ),
    None,
)
if _bootstrap_path is None:
    raise FileNotFoundError("Could not locate ecosystem notebooks/_bootstrap.py")

_bootstrap = runpy.run_path(str(_bootstrap_path))
repo_root = _bootstrap["repo_root"]
workspace_root = _bootstrap["workspace_root"]


# Interpolate Graph Demo
Load PubChem assay graphs and render a small sample.


In [ ]:
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2
import numpy as np
import scipy as sp
import pandas as pd
import networkx as nx
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display as ipy_display
from IPython.core.display import HTML
HTML('<style>.container { width:95% !important; }</style><style>.output_png {display: table-cell;text-align: center;vertical-align: middle;}</style>')

In [ ]:
from sklearn.metrics.pairwise import pairwise_kernels
import numpy as np
def plot_analysis(embeddings, force_ylim=True):
    K = pairwise_kernels(embeddings, metric='cosine')
    plt.imshow(K, cmap='gray')
    plt.show()
    
    size = K.shape[0]
    plt.plot(K[0], label='0/4')
    #plt.plot(K[size//4], label='1/4')
    #plt.plot(K[size//2], label='1/2')
    #plt.plot(K[size//4*3], label='3/4')
    plt.plot(K[-1], label='4/4')
    if force_ylim: 
        plt.ylim(-0.05,1.05)
        plt.yticks(np.arange(0,1.1,0.1))
    plt.grid(lw=.5)
    plt.legend()
    plt.show()

In [ ]:
from abstractgraph.operators import *
from abstractgraph_ml.feasibility import FeasibilityEstimatorFeatureCannotExist, FeasibilityEstimator, FeasibilityEstimatorNumberOfNodesInRange
from abstractgraph_generative.interpolate import InterpolationEstimator

from abstractgraph_ml.estimators import GraphEstimator
from sklearn.ensemble import RandomForestClassifier
from abstractgraph.vectorize import AbstractGraphTransformer

def fit_graph_estimator(
    graphs,
    targets,
    nbits,
    decomposition_function,
    *,
    transformer_n_jobs=-1,
    rf_n_jobs=-1,
):
    graph_transformer = AbstractGraphTransformer(
        nbits=nbits,
        decomposition_function=decomposition_function,
        n_jobs=transformer_n_jobs,
    )
    graph_estimator = GraphEstimator(
        estimator=RandomForestClassifier(n_estimators=300, n_jobs=rf_n_jobs),
        transformer=graph_transformer,
    )
    graph_estimator.fit(graphs, targets)
    return graph_estimator, graph_transformer

def fit_interpolation_estimator(graphs, nbits=15, decomposition_function=None): 
    df = neighborhood(radius=1)
    fe1 = FeasibilityEstimatorFeatureCannotExist(decomposition_function=df, nbits=19)
    df = compose(cycle(), unlabel())
    fe2 = FeasibilityEstimatorFeatureCannotExist(decomposition_function=df, nbits=19)
    df = compose(neighborhood(radius=3), unlabel())
    fe3 = FeasibilityEstimatorFeatureCannotExist(decomposition_function=df, nbits=19)
    min_size, max_size = np.quantile([len(g) for g in graphs], [0.25, 0.75])
    fe4 = FeasibilityEstimatorNumberOfNodesInRange(min_size=min_size, max_size=max_size)
    feasibility_estimators = [fe1, fe2, fe3, fe4]
    feasibility_estimator = FeasibilityEstimator(feasibility_estimators)
    feasibility_estimator.fit(graphs)

    graph_transformer = AbstractGraphTransformer(
        nbits=nbits,
        decomposition_function=decomposition_function,
        return_dense=True,
    )
    interpolation_estimator = InterpolationEstimator(
        graph_transformer=graph_transformer,
        n_iterations=2,
        n_samples=10,
        k=5,
        same_node_size=False,
        feasibility_estimator=feasibility_estimator,
    )
    interpolation_estimator.fit(graphs)
    return interpolation_estimator

def iterated_interpolation(source_orig, destination_orig, interpolation_estimator, n_iterations=3):
    total_interpolations_head = []
    total_interpolations_tail = []

    interpolated_graphs = interpolation_estimator.interpolate(source_orig, destination_orig)

    total_interpolations_head += interpolated_graphs[:len(interpolated_graphs)//2-1]
    total_interpolations_tail += interpolated_graphs[len(interpolated_graphs)//2:][::-1]

    for i in range(n_iterations):
        source = interpolated_graphs[len(interpolated_graphs)//2-1]
        destination = interpolated_graphs[len(interpolated_graphs)//2]
        interpolated_graphs = interpolation_estimator.interpolate(source, destination)
        total_interpolations_head += interpolated_graphs[:len(interpolated_graphs)//2-1]
        total_interpolations_tail += interpolated_graphs[len(interpolated_graphs)//2:][::-1]
        
    total_interpolations = total_interpolations_head + total_interpolations_tail[::-1]
    from abstractgraph.hashing import GraphHashDeduper
    total_interpolations_dedup = GraphHashDeduper().fit([source_orig, destination_orig]).filter(total_interpolations)
    total_interpolations_dedup = GraphHashDeduper().fit_filter(total_interpolations_dedup)
    return total_interpolations_dedup


from scipy.stats import spearmanr

def interpolation_score(interpolated_graphs, graph_transformer):
    embeddings = graph_transformer.transform(interpolated_graphs)
    gram = embeddings @ embeddings.T
    n = gram.shape[0]
    if n < 2:
        return np.nan
    scores = []
    for i in range(n):
        sim_row = gram[i]
        # forward
        if i + 1 < n:
            forward_idx = np.arange(i + 1, n)
            forward_sims = sim_row[forward_idx]
            forward_dist = np.arange(1, n - i)
            corr = spearmanr(forward_dist, forward_sims).correlation
            if np.isfinite(corr):
                scores.append(-corr)
        # backward
        if i - 1 >= 0:
            backward_idx = np.arange(0, i)
            backward_sims = sim_row[backward_idx][::-1]
            backward_dist = np.arange(1, i + 1)
            corr = spearmanr(backward_dist, backward_sims).correlation
            if np.isfinite(corr):
                scores.append(-corr)
    if not scores:
        return np.nan
    return float(np.mean(scores))


In [ ]:
from sklearn.cluster import KMeans
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import minimum_spanning_tree, dijkstra
from scipy.spatial.distance import cdist

def _mutual_knn_graph(dist, k=3):
    n = dist.shape[0]
    adj = np.zeros_like(dist, dtype=float)
    k = min(k, max(0, n - 1))
    if k == 0 or n == 0:
        return csr_matrix(adj)
    neighbor_sets = []
    for i in range(n):
        order = np.argsort(dist[i])
        neighbors = []
        for j in order:
            if j == i:
                continue
            neighbors.append(j)
            if len(neighbors) == k:
                break
        neighbor_sets.append(set(neighbors))
    for i in range(n):
        for j in neighbor_sets[i]:
            if i in neighbor_sets[j]:
                adj[i, j] = dist[i, j]
                adj[j, i] = dist[i, j]
    return csr_matrix(adj)

def _merge_graphs(mst, knn):
    mst_arr = mst.toarray()
    knn_arr = knn.toarray()
    adj = np.zeros_like(mst_arr)
    mask_mst = mst_arr != 0
    mask_knn = knn_arr != 0
    adj[mask_mst] = mst_arr[mask_mst]
    adj[mask_knn] = np.where(
        adj[mask_knn] == 0, knn_arr[mask_knn], np.minimum(adj[mask_knn], knn_arr[mask_knn])
    )
    return csr_matrix(adj)

def _neighborhood_indices(dist, k_knn=3, k_neighbors=11):
    mst = minimum_spanning_tree(csr_matrix(dist))
    mst = mst + mst.transpose()
    knn = _mutual_knn_graph(dist, k=k_knn)
    adj = _merge_graphs(mst, knn)
    neighbor_indices = []
    for i in range(dist.shape[0]):
        path_dist = dijkstra(adj, directed=False, indices=i)
        order = np.argsort(path_dist)
        neighbors = [
            j for j in order if j != i and np.isfinite(path_dist[j])
        ][:k_neighbors]
        neighbor_indices.append(neighbors)
    return neighbor_indices, adj

def _binary_entropy(p, eps=1e-8):
    p = np.clip(p, eps, 1 - eps)
    return -(p * np.log(p) + (1 - p) * np.log(1 - p))

def estimate_neighborhood_entropy(
    graphs,
    targets,
    graph_estimator, 
    graph_transformer,
    *,
    k_knn=3,
    k_neighbors=11,
    n_clusters=10,
    return_full=False,
):
    embeddings = graph_transformer.transform(graphs)
    dist = cdist(embeddings, embeddings, metric='euclidean')
    neighbor_indices, adj = _neighborhood_indices(dist, k_knn=k_knn, k_neighbors=k_neighbors)
    entropies = []
    for neighbors in neighbor_indices:
        if not neighbors:
            entropies.append(np.nan)
            continue
        neighbor_graphs = [graphs[i] for i in neighbors]
        probs = graph_estimator.predict_proba(neighbor_graphs, log=False)[:, -1]
        mean_prob = np.mean(probs)
        entropies.append(_binary_entropy(mean_prob))
    entropies = np.array(entropies)
    clusterer = KMeans(n_clusters=n_clusters, n_init=10, random_state=0)
    clusters = clusterer.fit_predict(embeddings)
    if return_full:
        return entropies, embeddings, dist, neighbor_indices, adj, clusters
    return entropies, clusters

def select_pairs(entropies, clusters, probs, *, n_pairs=20, rng=None):
    entropies = np.asarray(entropies)
    clusters = np.asarray(clusters)
    probs = np.asarray(probs)
    valid = np.isfinite(entropies) & np.isfinite(probs)
    idx = np.where(valid)[0]
    if idx.size < 2:
        return []
    pareto = []
    for i in idx:
        dominated = False
        for j in idx:
            if i == j:
                continue
            if probs[j] >= probs[i] and entropies[j] <= entropies[i]:
                if probs[j] > probs[i] or entropies[j] < entropies[i]:
                    dominated = True
                    break
        if not dominated:
            pareto.append(i)
    if len(pareto) < 2:
        return []
    pareto = np.array(pareto)
    rng = np.random.default_rng() if rng is None else rng
    pairs = []
    seen = set()
    attempts = 0
    max_attempts = n_pairs * 50
    while len(pairs) < n_pairs and attempts < max_attempts:
        attempts += 1
        i, j = rng.choice(pareto, size=2, replace=False)
        if clusters[i] == clusters[j]:
            continue
        key = (i, j) if i < j else (j, i)
        if key in seen:
            continue
        seen.add(key)
        pairs.append(key)
    return pairs


---

In [ ]:
from abstractgraph_graphicalizer.chem import PubChemLoader

loader = PubChemLoader(on_error="skip")
from abstractgraph_graphicalizer.chem import draw_molecules

assay_ids = ['2631','624249','651741','588350','463230','492952','743219','492992','463213']
assay_id = assay_ids[4]

size = 200
use_equalized = True


limit_active = int(size // 2) if use_equalized else int(size)
limit_inactive = int(size // 2) if use_equalized else int(size)
graphs, targets = loader.load(
    assay_id,
    limit_active=limit_active,
    limit_inactive=limit_inactive,
)
targets = np.array(targets)
print(f"AID {assay_id} graphs: {len(graphs)}")

from abstractgraph.utils import plot_graph_label_counts
_ = plot_graph_label_counts(graphs, top=20, title='Dataset info', log_scale=True)

In [ ]:
nbits=14
df = add(cycle(), tree(), neighborhood(radius=(1,2)))

In [ ]:
%%time
graph_estimator, graph_transformer = fit_graph_estimator(graphs, targets, nbits=nbits, decomposition_function=df)

In [ ]:
%%time
entropies, clusters = estimate_neighborhood_entropy(
    graphs, targets, 
    graph_estimator, graph_transformer, 
    k_knn=3, k_neighbors=11, n_clusters=10)
probs = graph_estimator.predict_proba(graphs, log=False)[:, -1]
high_entropy_pairs = select_pairs(entropies, clusters, probs, n_pairs=5)

In [ ]:
%%time
interpolation_estimator = fit_interpolation_estimator(graphs, nbits=nbits, decomposition_function=df)

In [ ]:
# SELECT='random'
SELECT='high_entropy'

if SELECT=='random':
    source_idx, destination_idx = np.random.choice(len(graphs), size=2, replace=False)
elif SELECT=='high_entropy':
    source_idx, destination_idx = high_entropy_pairs[np.random.choice(len(high_entropy_pairs), size=1)[0]]
source_orig = graphs[source_idx]
destination_orig = graphs[destination_idx]
draw_molecules([source_orig,destination_orig], n_graphs_per_line=7)

In [ ]:
interpolated_graphs = iterated_interpolation(source_orig, destination_orig, interpolation_estimator, n_iterations=5)
draw_molecules([source_orig] + interpolated_graphs + [destination_orig], n_graphs_per_line=7)

In [ ]:
from nsppk import NSPPK 
embeddings = NSPPK(radius=1, distance=4, connector=1, nbits=15, parallel=True).fit_transform(interpolated_graphs)
interp_score = interpolation_score(interpolated_graphs, graph_transformer)
print(f'Interpolation smoothness score: {interp_score:.3f}')
plot_analysis(embeddings, force_ylim=False)

---

In [ ]:
from abstractgraph_ml.estimators import GraphEstimator
from sklearn.ensemble import RandomForestClassifier
from abstractgraph.vectorize import AbstractGraphTransformer

graph_estimator = GraphEstimator(
    estimator=RandomForestClassifier(n_estimators=100, n_jobs=-1),
    transformer=AbstractGraphTransformer(nbits=nbits, decomposition_function=df),
    )

n_repeats = 3
all_log_probs = []
for it in range(n_repeats):
    # bootstrap sample
    boot_idx = np.random.choice(len(graphs), size=len(graphs), replace=True)
    boot_graphs = [graphs[i] for i in boot_idx]
    boot_targets = targets[boot_idx]
    graph_estimator.fit(boot_graphs, boot_targets)
    log_probs = graph_estimator.predict_proba(interpolated_graphs, log=True)[:,-1]
    all_log_probs.append(log_probs)
log_probs_arr = np.array(all_log_probs)
q25, q75 = np.quantile(log_probs_arr, [0.25, 0.75], axis=0)
median = np.median(log_probs_arr, axis=0)
x = np.arange(len(median))

plt.figure(figsize=(10,5))
plt.fill_between(x, q25, q75, alpha=0.2)
plt.plot(x, median, linewidth=2.5)
plt.grid()
plt.show()


---